# Tier 2 — move the task capacity into the backbone

**Purpose: find out whether Surya's *pretrained* representation contains filament
chirality, by shrinking the head until it cannot memorise the training set.**

Tier 1 (`noise_floor_runs.ipynb`) could not ask that question. Its trajectory
(`runs/noise_floor_seed42/version_3/metrics.csv`) shows why:

| | value |
|---|---|
| `train_loss`, epoch 0 → 3 | 1.01 → **0.085** |
| `val_loss` across 5 epochs | 0.2605, 0.2587, 0.2816, 0.2890, 0.2489 |
| prior-collapse floor | **0.2524** |
| `val accuracy`, epochs 1-4 | **0.4545** = 5/11 = always-dextral |

Train loss fell by 12x while validation sat on the floor and accuracy never left the
constant-predictor value. That is **memorisation of 23 samples**, not slow convergence — so
more epochs or a larger learning rate alone would only overfit faster. The Tier 1 head had
1,642,241 parameters for 23 training examples; it can fit them regardless of what the
backbone says, which means the run measured the head's capacity rather than Surya's
representation.

This notebook redistributes almost the same parameter budget:

| | Tier 1 | Tier 2 |
|---|---|---|
| LoRA adapters (inside the backbone) | 1,515,520 | **3,031,040** |
| fine-tuning head (on top) | 1,642,241 | **1,281** |
| total trainable | 3,157,761 | **3,032,321** |
| fraction in the backbone | 48% | **99.96%** |

Same order of magnitude, almost all of it now *inside* the pretrained features. The
question becomes crisp: **can a 1,281-parameter readout find chirality, if the adapters are
allowed to reshape what it reads?**

Everything else is held at Tier 1 so the comparison means something: same catalog, same
`split` column, same `batch_size: 1`, same `deterministic: warn`, same seed 42.

## Set your cuda visible device

**IMPORTANT:** Since we are sharing resources, please make sure that the cuda visible device you put here is the one assigned to your team and your machine.

In [1]:
import os
# This machine exposes a single L40S (44 GiB) as device 0 — `nvidia-smi -L` to confirm on
# yours. Setting a device index that does not exist leaves torch with zero visible GPUs and
# Lightning's accelerator="auto" silently falls back to CPU, which at 4096x4096 x 13
# channels will never finish.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import gc
import math          # cosine schedule (see "Learning-rate schedule" below) — not in Tier 1
import sys
from pathlib import Path

import pandas as pd
import torch
import wandb
import yaml

import lightning as L
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger
from torch.utils.data import DataLoader

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks
from workshop_infrastructure.utils import apply_peft_lora
torch.set_float32_matmul_precision('medium')

## Load configuration

The same `configs/config_script.yaml` Tier 1 used, **left on its Tier 1 values**. The Tier 2
knobs are applied to the loaded object in a cell below, not by editing the YAML, for two
reasons: `noise_floor_runs.ipynb` stays reproducible from the same file, and the next cell
can assert what the baseline was before changing it.

In [3]:
# The config is the single source of truth. load_filament_config() parses it into a typed
# object, exactly as the training script does, so the same YAML behaves identically here.
from downstream_apps.filament_kyle.configs import load_filament_config

cfg = load_filament_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")

Loaded config for job: filament_characterization


## Download assets

The config says where the assets belong, so it is loaded first. `ensure_assets()` fetches only what is missing from HuggingFace, so re-running this is free.

In [4]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# Fine-tuning needs the pretrained backbone as well (~1.8 GB).
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers", "weights"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")

Loaded scalers for 13 channels.


## The Tier 2 intervention

Six knobs change. The cell below first **asserts the YAML is still at its Tier 1 values**,
so an unrelated edit to `config_script.yaml` cannot quietly move the baseline this run is
compared against, then applies the overrides in memory.

| knob | Tier 1 | Tier 2 | why |
|---|---|---|---|
| `model.penultimate_linear_layer` | `True` | **`False`** | removes 1,639,680 of the head's 1,642,241 parameters. The head becomes a single `Linear(1280, 1)`. This is the anti-memorisation knob. |
| `model.pooling` | `class_token` | **`global_average`** | `class_token` prepends a *randomly initialised* token that the pretrained backbone has never seen — it has to be learned from 23 samples. Averaging the 65,536 patch tokens reads the pretrained representation directly and adds **zero** parameters. This is the backbone knob, and the one I expect to matter most. |
| `model.dropout` | `0.2` | **`0.3`** | slightly stronger regularisation on the one remaining linear. |
| `lora_config.r` | `8` | **`16`** | with the head now tiny, the adapters are where task capacity has to live. |
| `lora_config.lora_alpha` | `8` | **`32`** | LoRA scales its update by `alpha / r`: Tier 1 was `8/8 = 1.0`, this is `32/16 = 2.0`. Doubling it is what lets the adapters actually move within ~120 optimizer steps. |
| `training.learning_rate` | `1e-4` | **`3e-4`** | paired with the warmup + cosine schedule below. On its own a 3x learning rate would just overfit faster; the point is that the thing being fitted is now the backbone. |

**Interaction worth knowing about.** With `class_token` pooling, `head_linear` was applied to
a `(B, 1, D)` tensor — one token. With `global_average` it would be applied to
`(B, 65536, D)`, i.e. a 1280x1280 matmul across every patch. Turning
`penultimate_linear_layer` off is therefore both the regularisation choice *and* what keeps
the two runs comparable in cost. (In practice the compute is negligible either way — see the
runtime note below, this run is I/O-bound — but the parameter count is not.)

**Deliberately unchanged:** `use_lora: True`, `freeze_backbone: False`, `lora_dropout: 0.1`,
`target_modules`, `batch_size: 1`, `deterministic: warn`, `seed: 42`, and the catalog.

In [5]:
# --- 1. Assert the baseline, THEN override -----------------------------------------------
# This is the Tier 1 specification, copied from noise_floor_runs.ipynb. If the YAML has
# drifted, the Tier 1 numbers in runs/noise_floor_summary.csv no longer describe the config
# this run is being compared against, and the comparison is meaningless. Fail loudly.
TIER1_SPEC = {
    "model.pooling":                    ("class_token", lambda: cfg.model.pooling),
    "model.penultimate_linear_layer":   (True,          lambda: cfg.model.penultimate_linear_layer),
    "model.dropout":                    (0.2,           lambda: cfg.model.dropout),
    "model.use_lora":                   (True,          lambda: cfg.model.use_lora),
    "model.freeze_backbone":            (False,         lambda: cfg.model.freeze_backbone),
    "model.lora_config.r":              (8,             lambda: cfg.model.lora_config.r),
    "model.lora_config.lora_alpha":     (8,             lambda: cfg.model.lora_config.lora_alpha),
    "model.lora_config.target_modules": (["fc1", "fc2", "attn.qkv", "attn.proj"],
                                                        lambda: cfg.model.lora_config.target_modules),
    "training.learning_rate":           (0.0001,        lambda: cfg.learning_rate),
    "training.batch_size":              (1,             lambda: cfg.batch_size),
    "training.deterministic":           ("warn",        lambda: cfg.deterministic),
}

drift = [f"{k}: Tier 1 was {want!r}, YAML now has {get()!r}"
         for k, (want, get) in TIER1_SPEC.items() if want != get()]
assert not drift, (
    "config_script.yaml has drifted from the Tier 1 spec, so runs/noise_floor_summary.csv\n"
    "does not describe the baseline for this run:\n  " + "\n  ".join(drift)
)
print("config_script.yaml still matches the Tier 1 spec — baseline is valid.\n")

# --- 2. The Tier 2 overrides -------------------------------------------------------------
# Applied to the loaded object, not the file. Nothing here is written back to disk, so
# noise_floor_runs.ipynb keeps reproducing Tier 1 from the same YAML.
#
# NOTE: mutating the config after load() bypasses the cross-field validation in
# ModelConfig.__post_init__ / TrainingConfig.__post_init__. None of the six knobs below
# participate in those invariants (they concern img_size/patch_size, spectral_blocks/depth,
# time_dim, and learned_flow vs deterministic), so that is safe here — but if you add an
# override of your own, check configs.py before assuming the same.
OVERRIDES = {
    "model.penultimate_linear_layer": False,
    "model.pooling":                  "global_average",
    "model.dropout":                  0.3,
    "model.lora_config.r":            16,
    "model.lora_config.lora_alpha":   32,
    "training.learning_rate":         3e-4,
}

print(f"  {'knob':38s} {'Tier 1':>16s}  ->  Tier 2")
print(f"  {'-' * 38} {'-' * 16}      {'-' * 16}")
for key, new in OVERRIDES.items():
    old = TIER1_SPEC[key][0]
    print(f"  {key:38s} {old!r:>16s}  ->  {new!r}")

cfg.model.penultimate_linear_layer = OVERRIDES["model.penultimate_linear_layer"]
cfg.model.pooling                 = OVERRIDES["model.pooling"]
cfg.model.dropout                 = OVERRIDES["model.dropout"]
cfg.model.lora_config.r           = OVERRIDES["model.lora_config.r"]
cfg.model.lora_config.lora_alpha  = OVERRIDES["model.lora_config.lora_alpha"]
cfg.learning_rate                 = OVERRIDES["training.learning_rate"]

# Predicted trainable counts, so a surprise in the model-build output is obvious rather
# than something you have to notice. Adapter arithmetic, for r = 16:
#   8 x attn.qkv   Linear(1280, 3840): A 16x1280 + B 3840x16  =  20480 +  61440
#   8 x attn.proj  Linear(1280, 1280): A 16x1280 + B 1280x16  =  20480 +  20480
#  10 x mlp.fc1    Linear(1280, 5120): A 16x1280 + B 5120x16  =  20480 +  81920
#  10 x mlp.fc2    Linear(5120, 1280): A 16x5120 + B 1280x16  =  81920 +  20480
# and the head is now only head_unembed: Linear(1280, 1) = 1280 + 1.
EXPECT_ADAPTERS = 8 * (20480 + 61440) + 8 * (20480 + 20480) + 10 * (20480 + 81920) + 10 * (81920 + 20480)
EXPECT_HEAD = 1280 + 1
print(f"\nexpected trainable: {EXPECT_ADAPTERS:,} adapters + {EXPECT_HEAD:,} head"
      f" = {EXPECT_ADAPTERS + EXPECT_HEAD:,}")
print(f"  (Tier 1 was 1,515,520 + 1,642,241 = 3,157,761)")

# With pooling != class_token no head_cls_token is built, and discover_head_modules() skips
# parameter-free children, so head_dropout is not wrapped either. modules_to_save should
# come out as exactly ['head_unembed'] — watch for that line in the LoRA output.
print("expected modules_to_save: ['head_unembed']")

config_script.yaml still matches the Tier 1 spec — baseline is valid.

  knob                                             Tier 1  ->  Tier 2
  -------------------------------------- ----------------      ----------------
  model.penultimate_linear_layer                     True  ->  False
  model.pooling                             'class_token'  ->  'global_average'
  model.dropout                                       0.2  ->  0.3
  model.lora_config.r                                   8  ->  16
  model.lora_config.lora_alpha                          8  ->  32
  training.learning_rate                           0.0001  ->  0.0003

expected trainable: 3,031,040 adapters + 1,281 head = 3,032,321
  (Tier 1 was 1,515,520 + 1,642,241 = 3,157,761)
expected modules_to_save: ['head_unembed']


## Run settings

**Runtime, measured rather than guessed.** From `runs/noise_floor_seed42/version_3/`, whose
`metrics.csv` was created at 19:56:56 and last written at 20:17:40: **1244 s for 5 epochs**
over 34 sample-passes per epoch (23 train + 11 val) = **249 s/epoch = 7.3 s per sample**,
plus roughly 70 s of fixed startup (1.8 GB checkpoint load, LoRA, worker spawn).

| epochs | optimizer steps (accum 2) | estimated wall |
|---|---|---|
| 2 (`SMOKE_TEST`) | 24 | ~10 min |
| 5 | 60 | ~22 min |
| **10 (this run)** | **120** | **~43 min** |
| 20 | 240 | ~84 min |

**Why none of the model knobs cost anything.** The S3 cache lives on EFS and one sample is
~1 GB; 1 GB at a typical ~140 MB/s single-stream EFS read is ~7.2 s, which is essentially
the whole 7.3 s. This run is **I/O-bound, not compute-bound**, so pooling, LoRA rank, head
size and learning rate are all free in wall-clock terms. Epochs x samples is the only lever
— which is also why `num_workers` is worth raising (see below).

Two caveats on the 1244 s: 229 unrelated files were downloading into the same cache during
that run, so treat it as +/-30%; and it was measured at `accum 4`, but accumulation changes
only how gradients are grouped, not how many forward passes happen, so `accum 2` costs the
same.

`SMOKE_TEST` defaults to **False** here, unlike Tier 1. The initial-prediction probe below
reports within ~2 minutes of starting, which catches the failure mode a smoke test would
have caught (a dead or mis-scaled readout), so spending 10 minutes to then spend 43 more is
usually not worth it. Set it True if you have changed the run function itself.

**WandB**: resolved up front rather than failing inside `trainer.fit()`. With no API key it
switches to `WANDB_MODE=offline` (upload later with `wandb sync wandb/offline-run-*`). To log
online, run `wandb login` in a **terminal** — a notebook cell cannot answer the API-key
prompt, which is what raises `StdinNotImplementedError`. `USE_WANDB = False` skips it
entirely; the CSV logger runs either way.

In [ ]:
SEEDS = [42]                          # Tier 1's seed, so the comparison is like-for-like

SMOKE_TEST = False                    # <-- True = 2 epochs, to validate the loop only
MAX_EPOCHS = 2 if SMOKE_TEST else 10
USE_WANDB = False                     # <-- set True once `wandb login` has been run in a terminal

# --- Optimizer-step budget ---------------------------------------------------------------
# Tier 1 used 4. Halving it doubles the number of optimizer steps per epoch (6 -> 12) at
# *identical* wall time, because the forward/backward count is unchanged — only the grouping
# of gradients differs. 2 still averages two samples per step, so this is not bare
# batch_size=1 noise, and the catalog is now near-balanced (11 sinistral / 12 dextral in
# train) so the 7:1 imbalance that motivated 4 in Tier 1 is gone.
ACCUMULATE_GRAD_BATCHES = 2

# --- Dataloader workers ------------------------------------------------------------------
# Raised from Tier 1's 4. This CANNOT change the result: the sample order is fixed by the
# sampler and its dedicated generator, and with data.drop_hmi_probability = 0.0 nothing in
# FilamentDataset.__getitem__ draws a random number, so no per-worker RNG is consumed. It
# only changes wall time — and since the run is EFS-read-bound at roughly 140 MB/s aggregate
# against 8 x ~150 MB/s of available prefetch, this is the largest free speedup on offer.
# (If you ever set drop_hmi_probability > 0, this stops being result-neutral.)
NUM_WORKERS = 8

# --- Learning-rate schedule --------------------------------------------------------------
# Fraction of total optimizer steps spent warming up from 0 to cfg.learning_rate. Tier 1's
# per-step trace oscillated between 0.25 and 0.39 in epoch 1 — single-sample gradients at
# full learning rate from step 0. Warmup is the standard fix; 10% of 120 steps is 12.
WARMUP_FRAC = 0.10

# --- Head initialisation -----------------------------------------------------------------
# head_unembed is now the entire head (1,281 parameters), reading the mean of 65,536
# LayerNorm'd tokens.
#
# The BIAS is initialised to the training class prior instead of zero, which is the change
# that makes this run readable: the model then *starts* at exactly the prior-collapse
# solution (val MSE 0.2524), so any val_loss below that is unambiguously something the prior
# alone cannot explain. Tier 1 started near 0 and spent its first steps travelling to the
# prior, which is why its early epochs were uninterpretable.
#
# The WEIGHT scale needs re-checking rather than inheriting Tier 1's 0.01. That value was
# tuned for the class-token path, where head_unembed read a single token; averaging 65,536
# tokens gives a smaller-magnitude input, so 0.01 risks a readout too small to produce any
# gradient at all. 0.1 is the starting guess and the probe in run_one_config() measures
# whether it was right — read its output before letting the run continue.
#
# Not zero, deliberately: a zero weight also zeroes the gradient to everything upstream
# (every LoRA adapter) for the first optimizer step.
HEAD_INIT_SCALE = 0.1
PROBE_INIT = True                     # one no-grad forward on 4 val samples, ~30 s

# --- The prior, computed from the catalog rather than hardcoded --------------------------
_cat = pd.read_csv(cfg.data.filament_index_path)
_train_y = (_cat.loc[_cat["split"] == "train", "chirality"].astype(float)
            if "split" in _cat.columns else _cat["chirality"].astype(float))
TRAIN_PRIOR = float(_train_y.mean())

if USE_WANDB:
    # Also silences wandb's "Failed to detect the name of this notebook" error.
    os.environ.setdefault("WANDB_NOTEBOOK_NAME", "tier2_backbone_lr_lora.ipynb")
    if wandb.api.api_key is None:
        os.environ["WANDB_MODE"] = "offline"
        print("wandb: no API key found -> WANDB_MODE=offline "
              "(sync later with `wandb sync wandb/offline-run-*`)")
    else:
        print(f"wandb: API key found -> logging online to project {cfg.wandb_project!r}")
else:
    print("wandb: disabled (USE_WANDB = False) -> CSV logger only, results still recorded")

# Accumulated across runs, at module scope so a crash does not lose completed runs.
results = []

print(f"\nseeds          : {SEEDS}")
print(f"max_epochs     : {MAX_EPOCHS}" + ("   (SMOKE TEST — 2 epochs, not the real run)" if SMOKE_TEST else ""))
print(f"accum / workers: {ACCUMULATE_GRAD_BATCHES} / {NUM_WORKERS}")
print(f"learning_rate  : {cfg.learning_rate}  (warmup {WARMUP_FRAC:.0%} + cosine to 0)")
print(f"pooling / head : {cfg.model.pooling} / penultimate_linear_layer={cfg.model.penultimate_linear_layer}")
print(f"lora           : r={cfg.model.lora_config.r}, alpha={cfg.model.lora_config.lora_alpha}"
      f" (scaling {cfg.model.lora_config.lora_alpha / cfg.model.lora_config.r:.1f})")
print(f"deterministic  : {cfg.deterministic!r}")
print(f"train prior    : p = {TRAIN_PRIOR:.4f}  -> head_unembed.bias initialised here")
print(f"train/val      : split comes from the `split` column of {os.path.basename(cfg.data.filament_index_path)}")

## Learning-rate schedule

`FlareLightningModule.configure_optimizers()` returns a bare Adam with a constant learning
rate, which is the right default for the shared template. This run needs warmup and decay,
so it is subclassed **here in the notebook** rather than by editing
`downstream_apps/template/lightning_modules/pl_simple_baseline.py` — a Tier 2 experiment
should not change what every other app inherits. If the schedule turns out to be what makes
this task work, that is the moment to promote it into the shared module.

Two details:

- `interval: "step"` means the scheduler advances once per **optimizer** step, which under
  `accumulate_grad_batches` is not once per batch. `total_steps` must therefore be counted
  in optimizer steps — `run_one_config()` computes it with `ceil`, not `//` (Tier 1's
  notebook printed `23 // 4 = 5` steps/epoch, but Lightning actually ran
  `ceil(23/4) = 6`, which is why its last checkpoint was `epoch=4-step=30`).
- The optimizer is still built over `self.parameters()`, exactly as the parent does, rather
  than filtering to `requires_grad`. Frozen parameters never receive a gradient so Adam
  allocates no state for them and the result is identical; keeping the call identical to
  Tier 1's removes one more thing that could explain a difference.

In [ ]:
from downstream_apps.template.lightning_modules.pl_simple_baseline import FlareLightningModule


class CosineWarmupLightningModule(FlareLightningModule):
    """FlareLightningModule with linear warmup then cosine decay to zero.

    Args:
        total_steps: Total number of *optimizer* steps in the run
            (= ceil(batches_per_epoch / accumulate_grad_batches) * max_epochs).
        warmup_steps: Optimizer steps spent ramping the learning rate from ~0 to ``lr``.
    """

    def __init__(self, *args, total_steps: int, warmup_steps: int, **kwargs):
        super().__init__(*args, **kwargs)
        self.total_steps = int(total_steps)
        self.warmup_steps = int(warmup_steps)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)

        def lr_scale(step: int) -> float:
            # LambdaLR multiplies the base lr by this factor. `step` is the number of
            # scheduler advances so far, i.e. optimizer steps, starting at 0.
            if step < self.warmup_steps:
                return (step + 1) / max(1, self.warmup_steps)
            decay_steps = max(1, self.total_steps - self.warmup_steps)
            progress = min(1.0, (step - self.warmup_steps) / decay_steps)
            return 0.5 * (1.0 + math.cos(math.pi * progress))

        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_scale)
        return {
            "optimizer": optimizer,
            # "step", not "epoch": with 12 optimizer steps per epoch a per-epoch schedule
            # would only move 10 times over the whole run.
            "lr_scheduler": {"scheduler": scheduler, "interval": "step", "frequency": 1},
        }

## Run one configuration

Structurally identical to `run_one_seed()` in `noise_floor_runs.ipynb` — same ordering, same
seeding-first rule, same teardown — with four additions, each marked in the code:

1. **`ceil` instead of `//`** when counting optimizer steps, so the printed budget matches
   what Lightning actually runs.
2. **`head_unembed.bias = TRAIN_PRIOR`**, so the run starts at the prior-collapse solution.
3. **The initial-prediction probe**, which runs one no-grad forward over 4 validation
   samples and prints the initial prediction mean/std and MSE *before* the long part starts.
   It probes `val_loader`, not `train_loader`: validation is built with `shuffle=False` and
   no generator, so iterating it consumes no RNG and cannot perturb the training shuffle.
4. **`LearningRateMonitor`**, so the schedule lands in `metrics.csv` and the trajectory cell
   can show it.

**What to look for in the probe**, in the first ~2 minutes:

- `std ~ 0` → the readout is effectively dead; raise `HEAD_INIT_SCALE` and restart.
- `mean` far from `TRAIN_PRIOR` → the bias init did not take, or the weight term dominates
  the bias; lower `HEAD_INIT_SCALE`.
- `MSE` much above the 0.2524 floor → the run is starting *worse* than a constant, which
  wastes the early optimizer steps; lower `HEAD_INIT_SCALE`.
- `mean ~ 0.48`, small `std`, `MSE ~ 0.25` → as intended. Let it run.

In [ ]:
from downstream_apps.filament_kyle.datasets.filament_dataset import FilamentDataset
# This app's own metrics module (val_metrics reports binary chirality accuracy alongside
# MSE and RRSE), as in Tier 1.
from downstream_apps.filament_kyle.metrics.template_metrics import FlareMetrics
from workshop_infrastructure.datasets.builders import build_helio_dataloaders
from workshop_infrastructure.models.finetune_models import HelioSpectformer1D
from workshop_infrastructure.utils import load_pretrained_weights


def probe_initial_predictions(model, loader, n_batches: int = 4):
    """Forward `n_batches` validation samples with no grad and return (preds, targets).

    Probes the validation loader on purpose: build_helio_dataloaders() gives validation
    shuffle=False and no generator, so creating a throwaway iterator over it consumes no
    RNG and cannot change the training shuffle order. Doing this on train_loader would.
    """
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(dev).eval()
    preds, targets = [], []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches:
                break
            # ds_index is a string, so move only the tensors.
            batch = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in batch.items()}
            # Match the trainer's precision="bf16-mixed" so the probe measures what
            # training will actually see.
            with torch.autocast(device_type=dev, dtype=torch.bfloat16, enabled=(dev == "cuda")):
                out = model(batch)
            preds.append(out.float().reshape(-1).cpu())
            targets.append(batch["forecast"].float().reshape(-1).cpu())
    model.train()   # leave the model where fit() expects it
    return torch.cat(preds), torch.cat(targets)


def run_one_config(seed: int) -> dict:
    """Train the Tier 2 configuration once at `seed` and return its validation numbers."""
    print(f"\n{'=' * 70}\n  Tier 2  |  seed {seed}  ({MAX_EPOCHS} epochs)\n{'=' * 70}")

    # 1. Seed FIRST — before the model is constructed, so head init and the LoRA A matrices
    #    are a function of `seed`. Same rule as Tier 1; the experiment depends on it.
    L.seed_everything(seed, workers=True)

    # 2. Dataloaders. NUM_WORKERS is the only change from Tier 1 here, and it cannot affect
    #    the result (see the run-settings cell).
    train_loader, val_loader = build_helio_dataloaders(
        cfg,
        FilamentDataset,
        scalers=scalers,
        num_workers=NUM_WORKERS,
        seed=seed,
        #### Downstream (DS) specific parameters
        return_surya_stack=True,
        max_number_of_samples=cfg.data.max_samples,
        filament_index_path=cfg.data.filament_index_path,
        ds_time_column=cfg.data.ds_time_column,
        ds_time_tolerance=cfg.data.ds_time_tolerance,
        ds_match_direction=cfg.data.ds_match_direction,
    )

    # 3. A fresh model. cfg.model now carries the Tier 2 overrides, so this builds the
    #    global_average / no-penultimate variant.
    model = HelioSpectformer1D.from_config(
        cfg.model,
        num_outputs=1,
        dtype=cfg.dtype,
        use_latitude_in_learned_flow=cfg.use_latitude_in_learned_flow,
    )
    load_pretrained_weights(model, cfg.model.pretrained_path)

    # --- ADDITION 2: start at the prior-collapse solution -------------------------------
    # bias = the training prior, so initial predictions sit at p for every input and the
    # run begins exactly at val MSE 0.2524. Any improvement is then unambiguous.
    #
    # This MUST happen before apply_peft_lora(): PEFT wraps head_unembed in a
    # ModulesToSaveWrapper that *copies* the module, and reaching through the wrapper
    # afterwards can hand back the frozen original rather than the trainable copy.
    with torch.no_grad():
        model.head_unembed.weight.mul_(HEAD_INIT_SCALE)
        model.head_unembed.bias.fill_(TRAIN_PRIOR)

    if cfg.model.freeze_backbone:
        for name, param in model.named_parameters():
            if name.startswith("backbone."):
                param.requires_grad = False
    if cfg.model.use_lora:
        model = apply_peft_lora(model, cfg.model.lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[tier2 seed {seed}] trainable parameters: {trainable:,}"
          f"   (expected {EXPECT_ADAPTERS + EXPECT_HEAD:,})")
    if trainable != EXPECT_ADAPTERS + EXPECT_HEAD:
        print(f"[tier2 seed {seed}] WARNING: trainable count is not what the override cell"
              " predicted — check the LoRA module list above before trusting this run.")

    # --- ADDITION 3: probe the initial readout before spending the wall time ------------
    if PROBE_INIT:
        p, t = probe_initial_predictions(model, val_loader)
        init_mse = float(((p - t) ** 2).mean())
        print(f"\n[tier2 seed {seed}] initial readout over {len(p)} val samples:")
        print(f"    pred mean {p.mean():.4f}  std {p.std(unbiased=False):.4f}"
              f"   (target: mean ~= TRAIN_PRIOR = {TRAIN_PRIOR:.4f}, std small)")
        print(f"    pred min  {p.min():.4f}   max {p.max():.4f}")
        print(f"    initial MSE on these samples = {init_mse:.4f}"
              f"   (prior-collapse floor is ~0.2524)")
        if p.std(unbiased=False) < 1e-4:
            print("    >> std is ~0: the readout is dead. Raise HEAD_INIT_SCALE and restart.")
        elif init_mse > 0.5:
            print("    >> starting well above the floor. Lower HEAD_INIT_SCALE and restart.")
        else:
            print("    >> looks sane — starting at roughly the prior. Letting it run.")
        print()

    # 4. Metrics and Lightning module. Rebuilt per run so the cached torchmetrics
    #    instances do not carry state across runs.
    metrics = {
        "train_loss": FlareMetrics("train_loss"),
        "val_loss": FlareMetrics("val_loss"),
        "train_metrics": FlareMetrics("train_metrics"),
        "val_metrics": FlareMetrics("val_metrics"),
    }

    # --- ADDITION 1: ceil, not //. Lightning takes a partial accumulation group at the end
    # of the epoch as a real optimizer step, so 23 batches at accum 2 is 12 steps, not 11.
    # Tier 1's notebook used // and under-reported its own budget by one step per epoch.
    opt_steps_per_epoch = math.ceil(len(train_loader) / ACCUMULATE_GRAD_BATCHES)
    total_opt_steps = opt_steps_per_epoch * MAX_EPOCHS
    warmup_steps = max(2, round(WARMUP_FRAC * total_opt_steps))
    print(f"[tier2 seed {seed}] {len(train_loader)} batches/epoch / accum"
          f" {ACCUMULATE_GRAD_BATCHES} = {opt_steps_per_epoch} optimizer step(s)/epoch,"
          f" {total_opt_steps} total ({warmup_steps} warmup)")

    lit_model = CosineWarmupLightningModule(
        model, metrics, lr=cfg.learning_rate, batch_size=cfg.batch_size,
        total_steps=total_opt_steps, warmup_steps=warmup_steps,
    )

    # 5. One logger set and one checkpoint directory per run.
    run_name = f"tier2_seed{seed}"
    loggers = [CSVLogger("runs", name=run_name)]
    if USE_WANDB:
        loggers.append(
            WandbLogger(
                entity=cfg.wandb_entity,
                project=cfg.wandb_project,
                name=run_name,
                log_model=False,
                save_dir="./wandb/wandb_tmp",
            )
        )

    checkpoint_cb = ModelCheckpoint(
        dirpath=Path("checkpoints/tier2") / run_name,
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )
    # ADDITION 4: writes the scheduled lr into metrics.csv, so the trajectory cell can show
    # whether warmup and decay actually happened.
    lr_cb = LearningRateMonitor(logging_interval="step")

    trainer = L.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator="auto",
        devices="auto",
        precision="bf16-mixed",
        logger=loggers,
        callbacks=[checkpoint_cb, lr_cb],
        # Passed through from training.deterministic ("warn"), as in Tier 1. Without this
        # the config key has no effect and a Tier 1 vs Tier 2 difference could be
        # nondeterminism rather than the intervention.
        deterministic=cfg.deterministic,
        accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES,
        # 1, because log_every_n_steps counts optimizer steps and there are only 12 per
        # epoch. Seeing the per-step trajectory is the point.
        log_every_n_steps=1,
    )
    trainer.fit(lit_model, train_loader, val_loader)

    # 6. Collect. best_val_loss is what ModelCheckpoint selected on; last_val_loss is the
    #    end-of-training value. With 11 validation samples the best epoch can be a fluke,
    #    and a large best-vs-last gap is itself a warning.
    m = trainer.callback_metrics

    def scalar(key):
        v = m.get(key)
        return float(v) if v is not None else float("nan")

    best = checkpoint_cb.best_model_score
    result = {
        "tier": 2,
        "seed": seed,
        "epochs": MAX_EPOCHS,
        "accum": ACCUMULATE_GRAD_BATCHES,
        "pooling": cfg.model.pooling,
        "penultimate": cfg.model.penultimate_linear_layer,
        "dropout": cfg.model.dropout,
        "lora_r": cfg.model.lora_config.r,
        "lora_alpha": cfg.model.lora_config.lora_alpha,
        "lr": cfg.learning_rate,
        "opt_steps": total_opt_steps,
        "trainable": trainable,
        "best_val_loss": float(best) if best is not None else float("nan"),
        "last_val_loss": scalar("val_loss"),
        "last_val_mse": scalar("val_metric_mse"),
        "last_val_rrse": scalar("val_metric_rrse"),
        "last_val_accuracy": scalar("val_metric_accuracy"),
        "last_train_loss": scalar("train_loss"),
    }
    print(f"[tier2 seed {seed}] best val_loss = {result['best_val_loss']:.6f} | "
          f"last val_loss = {result['last_val_loss']:.6f} | "
          f"last val accuracy = {result['last_val_accuracy']:.4f}")

    # 7. Free the GPU before anything else builds its own 366M-parameter copy.
    if USE_WANDB:
        wandb.finish()
    del trainer, lit_model, model, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## Run it

One run, ~43 minutes. The probe reports about two minutes in — **read it before walking
away**, since the three failure modes it catches all mean the remaining 41 minutes are
wasted.

The summary is written to `runs/tier2_summary.csv` after each run, so an interruption keeps
whatever finished.

In [ ]:
SUMMARY_CSV = Path("runs/tier2_summary.csv")
SUMMARY_CSV.parent.mkdir(parents=True, exist_ok=True)

for seed in SEEDS:
    results.append(run_one_config(seed))
    pd.DataFrame(results).to_csv(SUMMARY_CSV, index=False)

print(f"\n{len(results)} run(s) complete -> {SUMMARY_CSV}")

## The per-step trajectory

This is the cell that answers "how do I see the loss at each excursion" — every logged step
is already in `runs/<run_name>/version_*/metrics.csv`, the print output at the end of
training just does not show it. Reading it is how Tier 1 was diagnosed as memorisation
rather than under-training.

What to read here:

- **`train_loss` per step.** Under `accumulate_grad_batches` Lightning logs the *last
  micro-batch's* loss, not the accumulated average — so a single spike is often just one
  hard sample, not an optimizer excursion. Look at the trend across epochs, not individual
  values.
- **`lr`.** Should ramp over the first 12 steps then decay smoothly to ~0. If it is flat,
  the scheduler is not wired up.
- **`val_loss` per epoch against 0.2524.** The whole result.
- **`val_metric_accuracy`.** With 11 validation filaments this can only take values k/11.
  0.4545 = 5/11 = always-dextral; 0.5455 = 6/11 = always-sinistral (the val majority). A
  value pinned at either across every epoch means constant prediction, whatever the loss
  says.

In [ ]:
# Newest Tier 2 run directory, by mtime.
run_dirs = sorted(Path("runs").glob("tier2_seed*/version_*"), key=lambda p: p.stat().st_mtime)
assert run_dirs, "no runs/tier2_seed*/version_* found — has the training cell been run?"
latest = run_dirs[-1]
print(f"reading {latest}\n")

m = pd.read_csv(latest / "metrics.csv")

# Per-optimizer-step training rows: those with a train_loss value.
train_cols = [c for c in ["epoch", "step", "train_loss", "lr-Adam", "train_metric_rrse"] if c in m.columns]
tr = m.dropna(subset=["train_loss"])[train_cols]
print("per-step training trajectory")
print(tr.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# Per-epoch validation rows: those with a val_loss value.
val_cols = [c for c in ["epoch", "step", "val_loss", "val_metric_accuracy"] if c in m.columns]
va = m.dropna(subset=["val_loss"])[val_cols]
print("\nper-epoch validation")
print(va.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# Did the schedule actually run?
lr_col = next((c for c in m.columns if c.startswith("lr-")), None)
if lr_col is not None:
    lrs = m[lr_col].dropna()
    print(f"\nlearning rate: min {lrs.min():.3e}  max {lrs.max():.3e}"
          f"  first {lrs.iloc[0]:.3e}  last {lrs.iloc[-1]:.3e}")
    if lrs.max() - lrs.min() < 1e-12:
        print("  >> WARNING: lr never changed — the cosine scheduler did not take effect.")
else:
    print("\nno lr-* column found (LearningRateMonitor did not log).")

## Verdict

One number decides this run: does `best_val_loss` fall **below the prior-collapse floor by
more than the Tier 1 noise floor**?

Both references are read from disk rather than hardcoded — the floor is recomputed from the
catalog (the previously hardcoded 0.2656 was correct only for the old 8-train/7:1 split and
would have gone silently stale), and the Tier 1 comparison comes from
`runs/noise_floor_summary.csv`.

Be honest about the statistics: Tier 1 was run at a single seed, so there is **no measured
spread** to compare against, and `PRIOR_TOL` below is a stand-in rather than a real noise
floor. A Tier 2 result inside that tolerance is not evidence of anything. If Tier 2 does
come out clearly below the floor, the next thing to run is the *same* config at seeds 43 and
44 — that turns a suggestive number into a measurement.

In [ ]:
df = pd.DataFrame(results)
show = [c for c in ["seed", "epochs", "opt_steps", "trainable", "best_val_loss",
                    "last_val_loss", "last_val_accuracy", "last_train_loss"] if c in df.columns]
print(df[show].to_string(index=False, float_format=lambda x: f"{x:.6f}"))

s = df["best_val_loss"]
spread = float(s.max() - s.min())
mean_best = float(s.mean())

# --- Trivial-predictor reference, recomputed from the catalog ----------------------------
# Caveat: these labels come straight from the catalog, whereas the datasets additionally
# drop any event with no Surya frame inside data.ds_time_tolerance. The two agree only when
# every event matches — compare the counts below against the dataloader output above.
cat = pd.read_csv(cfg.data.filament_index_path)
if "split" in cat.columns:
    train_y = cat.loc[cat["split"] == "train", "chirality"].astype(float)
    val_y = cat.loc[cat["split"] == "val", "chirality"].astype(float)
else:
    train_y = val_y = cat["chirality"].astype(float)
    print("\n  NOTE: catalog has no `split` column; using all events for both references.")

prior = float(train_y.mean())
prior_collapse_val_mse = float(((val_y - prior) ** 2).mean())
always_dextral_acc = float((val_y == 0).mean())
majority_val_acc = float(max((val_y == 0).mean(), (val_y == 1).mean()))

print(f"\ntrivial-predictor reference ({len(cat)} catalog events,"
      f" {len(train_y)} train / {len(val_y)} val):")
print(f"  training class prior            p = {prior:.4f}")
print(f"  val MSE predicting only p         = {prior_collapse_val_mse:.4f}   <-- collapse floor")
print(f"  val accuracy always-dextral       = {always_dextral_acc:.4f}")
print(f"  val accuracy of majority class    = {majority_val_acc:.4f}")

# --- Tier 1 baseline, read from its own summary ------------------------------------------
TIER1_CSV = Path("runs/noise_floor_summary.csv")
tier1_best = float("nan")
if TIER1_CSV.exists():
    t1 = pd.read_csv(TIER1_CSV)
    tier1_best = float(t1["best_val_loss"].min())
    print(f"\nTier 1 baseline ({len(t1)} run(s) in {TIER1_CSV.name}):"
          f" best_val_loss = {tier1_best:.6f}")
    if len(t1) < 2:
        print("  NOTE: one Tier 1 run only — there is no measured seed spread to compare against.")
else:
    print(f"\nTier 1 baseline: {TIER1_CSV} not found; comparing against the floor only.")

# PRIOR_TOL is a placeholder for the unmeasured noise floor, not a measured one. With a
# single seed the spread is 0, so max(spread, PRIOR_TOL) is what makes this check fire at
# n=1 at all.
PRIOR_TOL = 0.02
tol = max(spread, PRIOR_TOL)

print(f"\n{'=' * 70}")
if mean_best < prior_collapse_val_mse - tol:
    print(f"  RESULT: best_val_loss {mean_best:.4f} is below the collapse floor"
          f" {prior_collapse_val_mse:.4f}")
    print(f"  by more than {tol:.4f}. Something beyond the class balance was learned.")
    print(f"  Confirm it two ways before believing it:")
    print(f"    1. last_val_accuracy must differ from {always_dextral_acc:.4f} and"
          f" {majority_val_acc:.4f}")
    print(f"       (a constant predictor can still post a decent MSE).")
    print(f"    2. re-run this config at seeds 43 and 44. One seed is not a measurement.")
elif abs(mean_best - prior_collapse_val_mse) < tol:
    print(f"  RESULT: best_val_loss {mean_best:.4f} is within {tol:.4f} of the collapse")
    print(f"  floor {prior_collapse_val_mse:.4f} — the run still learned the class balance")
    print(f"  and nothing about chirality. Shrinking the head did not help.")
    print(f"  Check last_val_accuracy: if it equals {always_dextral_acc:.4f} or"
          f" {majority_val_acc:.4f}, prediction is constant.")
else:
    print(f"  RESULT: best_val_loss {mean_best:.4f} is WORSE than the collapse floor"
          f" {prior_collapse_val_mse:.4f}")
    print(f"  — worse than predicting one constant. Suspect an unstable run rather than a")
    print(f"  hard task: check the per-step trajectory above for a diverging lr or a")
    print(f"  train_loss that never descends.")

if not math.isnan(tier1_best):
    delta = mean_best - tier1_best
    print(f"\n  vs Tier 1: {mean_best:.6f} - {tier1_best:.6f} = {delta:+.6f}")
    if abs(delta) < tol:
        print(f"  Within {tol:.4f} of Tier 1. The intervention changed nothing measurable —")
        print(f"  which, given it moved 99.96% of the trainable parameters into the backbone,")
        print(f"  is itself informative: the bottleneck is not head capacity.")

print(f"{'=' * 70}")
if SMOKE_TEST:
    print("\n  SMOKE_TEST is True — 2 epochs, NOT the real run. Set it False and re-run.")

## Conclusion — and what the two possible outcomes mean

**If `best_val_loss` drops clearly below 0.2524:** Surya's pretrained representation does
carry chirality information, and the Tier 1 result was a head-capacity artefact. Next step
is *not* another hyperparameter — it is seeds 43 and 44 on this exact config, to turn one
number into a measurement. After that, `global_max` pooling is the natural follow-up: a
filament is a localised structure, and max-pooling preserves a local activation that
averaging over 65,536 patches dilutes. (It was not the first choice here because its
gradient reaches far fewer tokens per step, and with 23 training samples variance is the
binding problem.)

**If it does not:** that is the more likely outcome, and it is a real finding rather than a
failed run. Moving 99.96% of the trainable parameters from the head into the backbone and
still landing on the prior means the limit is not head capacity, not LoRA rank, and not the
learning rate. With 23 training and 11 validation events, the honest next moves are about
the *data*, not the model:

- **more labelled filaments** — this is the dominant term, and nothing else competes with it;
- **leave-one-out cross-validation** over all 34 events, which turns an 11-sample validation
  estimate into a 34-sample one at 34x the compute (~24 h at the measured 249 s/epoch, so
  worth doing only once the per-run question is settled);
- **the Martin's Rule baseline** in `1_baseline_filament.ipynb` — the `hemisphere` column is
  deliberately withheld from the model, so if Martin's Rule beats every fine-tune, that is
  the honest headline result and worth reporting as one.

One thing this notebook does **not** establish either way: `deterministic: warn` makes a
single run reproducible, not statistically significant. Every number here is one draw.